# Projet ML - Classification binaire : **Diabetes Health Indicators**


Jeu de données BRFSS 2015 (Kaggle) : prédire si une personne est diabétique / pré-diabétique (`Diabetes_binary = 1`) à partir de 21 indicateurs de santé et de mode de vie. Le même workflow (`ml_workflow.py`) que pour Spambase est appliqué ; la **difficulté propre** à ce jeu est le **fort déséquilibre des classes** (~14 % de positifs) et sa **taille** (253 680 lignes).

> **Règle suivie** : choix de modèles / hyperparamètres / seuil par validation croisée sur le **train uniquement** ; le test n'est utilisé qu'à la fin. Pour aller vite, la validation croisée et le réglage se font sur un **sous-échantillon stratifié du train** ; les modèles finaux sont réentraînés sur le train complet.

In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import ml_workflow as ml

pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 100
SEED = ml.RANDOM_STATE
N_SUB = 30_000      # taille du sous-échantillon pour CV / réglage

## 1. Import et description des données  

In [3]:
# Loading
df_diabetes = ml.load_diabetes()

# what it looks like
display(df_diabetes.head())

# Dimension
print("Dimensions :", df_diabetes.shape)

# General description
display(
    ml.describe_dataset(
        df_diabetes,
        "Diabetes_binary"
    )
)

# Target distribution
display(
    df_diabetes["Diabetes_binary"].value_counts()
)

# percentage distribution
display(
    df_diabetes["Diabetes_binary"]
    .value_counts(normalize=True)
    .mul(100)
)


X_diabetes, y_diabetes = ml.split_features_target(
    df_diabetes,
    "Diabetes_binary"
)

print("X :", X_diabetes.shape)
print("y :", y_diabetes.shape)

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


Dimensions : (253680, 22)


n_rows                                                          253680
n_columns                                                           22
n_features                                                          21
n_missing_values                                                     0
n_duplicated_rows                                                24206
target                                                 Diabetes_binary
class_counts                                 {0.0: 218334, 1.0: 35346}
class_proportions    {0.0: 0.8606669820245979, 1.0: 0.1393330179754...
dtype: object

Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64

Diabetes_binary
0.0    86.066698
1.0    13.933302
Name: proportion, dtype: float64

X : (253680, 21)
y : (253680,)


## 2. Pré-traitement  

In [4]:
# MCAR simulation of 5% missing values
X_diabetes_missing = ml.inject_missing_values(
    X_diabetes,
    frac=0.05,
    seed=SEED
)

# Stratified separation
X_train_diabetes, X_test_diabetes, y_train_diabetes, y_test_diabetes = ml.split_data(
    X_diabetes_missing,
    y_diabetes,
    test_size=0.2,
    seed=SEED
)

print("Train :", X_train_diabetes.shape)
print("Test :", X_test_diabetes.shape)

print("\nDistribution du train :")
display(y_train_diabetes.value_counts(normalize=True))

print("\nDistribution du test :")
display(y_test_diabetes.value_counts(normalize=True))

print("\nValeurs manquantes train :")
print(X_train_diabetes.isna().sum().sum())

print("\nValeurs manquantes test :")
print(X_test_diabetes.isna().sum().sum())

# Subsample for future cross-validation operations
X_sub, y_sub = ml.subsample(
    X_train_diabetes,
    y_train_diabetes,
    n=N_SUB,
    seed=SEED
)

print("\nSous-échantillon :", X_sub.shape)

display(y_sub.value_counts(normalize=True))

Train : (202944, 21)
Test : (50736, 21)

Distribution du train :


Diabetes_binary
0.0    0.860666
1.0    0.139334
Name: proportion, dtype: float64


Distribution du test :


Diabetes_binary
0.0    0.860671
1.0    0.139329
Name: proportion, dtype: float64


Valeurs manquantes train :
213153

Valeurs manquantes test :
53211

Sous-échantillon : (30000, 21)


Diabetes_binary
0.0    0.860667
1.0    0.139333
Name: proportion, dtype: float64

## 3. Gestion du déséquilibre  
*(responsable : **personne C**)*

In [6]:
# TODO personne C
print(y_train_diabetes.value_counts(normalize=True).round(3))

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
dummy = Pipeline([("model", DummyClassifier(strategy="prior"))]).fit(X_train_diabetes, y_train_diabetes)
base = ml.evaluate_model(dummy, X_test_diabetes, y_test_diabetes)   # roc_auc = 0.5, pr_auc = prévalence
pd.Series(base, name="Dummy (toujours non)").round(3)

Diabetes_binary
0.0    0.861
1.0    0.139
Name: proportion, dtype: float64


accuracy             0.861
balanced_accuracy    0.500
precision            0.000
recall               0.000
f1                   0.000
mcc                  0.000
roc_auc              0.500
pr_auc               0.139
Name: Dummy (toujours non), dtype: float64

## 4. Comparaison de plusieurs méthodes (CV sur le sous-échantillon, classes pondérées)  

In [7]:
models = ml.get_models(class_weight="balanced", include_svm=False, random_state=SEED)
cv_table = ml.cross_validate_models(
    models,
    X_sub,
    y_sub,
    cv=5,
    scoring=("f1", "roc_auc", "average_precision"),
)
display(cv_table.round(3).sort_values("average_precision_mean", ascending=False))

,f1_mean,f1_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,fit_time_s
hgb,0.436,0.006,0.820,0.009,0.416,0.023,0.537
lr,0.436,0.010,0.817,0.010,0.398,0.019,0.145
rf,0.176,0.017,0.794,0.007,0.360,0.014,2.168


## 5. Sélection de variables  

In [8]:
selector_model = LogisticRegression(
    class_weight="balanced", max_iter=1000, random_state=SEED
)
selector_table = ml.compare_feature_selectors(
    selector_model,
    X_sub,
    y_sub,
    methods=("anova", "mutual_info", "l1", "rf"),
    k_values=(5, 10, 15),
    cv=5,
    scoring="average_precision",
)
display(selector_table.round(3).sort_values("average_precision_mean", ascending=False))

pca_table = ml.compare_pca(
    selector_model,
    X_sub,
    y_sub,
    pca_components=(None, 0.95),
    cv=5,
    scoring="average_precision",
)
display(pca_table.round(3).sort_values("average_precision_mean", ascending=False))

average_precision_mean  average_precision_std  fit_time_s
method      k                                                            
l1          15                   0.398                  0.018       0.183
            10                   0.397                  0.019       0.187
anova       15                   0.396                  0.018       0.104
mutual_info 15                   0.396                  0.018       3.065
anova       10                   0.394                  0.018       0.100
mutual_info 10                   0.394                  0.018       2.377
rf          15                   0.391                  0.019       3.237
l1          5                    0.389                  0.018       0.215
mutual_info 5                    0.387                  0.019       1.590
anova       5                    0.384                  0.017       0.092
rf          10                   0.377                  0.016       1.801
            5                    0.359                  0.018       1.748

,average_precision_mean,average_precision_std,fit_time_s
none,0.398,0.019,0.167
0.95,0.391,0.019,0.159


## 6. Réglage des hyperparamètres de HistGradientBoosting (sous-échantillon, critère PR-AUC)  

In [9]:
hgb = HistGradientBoostingClassifier(
    class_weight="balanced", random_state=SEED
)
hgb_pipe = ml.build_pipeline(hgb)
hgb_grid = ml.tune_model(
    hgb_pipe,
    {
        "model__learning_rate": [0.03, 0.1],
        "model__max_iter": [100, 200],
        "model__l2_regularization": [0.0, 1.0],
    },
    X_train_diabetes,
    y_train_diabetes,
    scoring="average_precision",
    cv=3,
    n_samples=N_SUB,
)
print("Meilleurs hyperparamètres :", hgb_grid.best_params_)
print("Meilleur score PR-AUC (CV sur sous-échantillon) :", round(hgb_grid.best_score_, 3))

Meilleurs hyperparamètres : {'model__l2_regularization': 1.0, 'model__learning_rate': 0.1, 'model__max_iter': 100}
Meilleur score PR-AUC (CV sur sous-échantillon) : 0.414


## 7. Entraînement final sur le train complet et évaluation sur le test  
*(responsable : **personne C**)*

In [ ]:
# TODO personne C
cv_table = ml.cross_validate_models(models, X_train, y_train, cv=5)
cv_table.round(3).sort_values("average_precision_mean", ascending=False)

In [ ]:
# TODO personne C
# 
final = ml.build_pipeline(best_model, **best_params)       # adapter
oof   = ml.oof_scores(final, X_train, y_train, cv=3)
thr   = ml.find_best_threshold(y_train, oof, metric="f1")
final.fit(X_train, y_train)

res = pd.DataFrame({
    "seuil par défaut": ml.evaluate_model(final, X_test, y_test),
    f"seuil optimisé ({thr:.2f})": ml.evaluate_model(final, X_test, y_test, threshold=thr),
}).round(3)
res

## 8. Conclusions — Diabetes Health Indicators  
*(responsable : **personne C**)*

In [ ]:
# TODO personne C